In [2]:
!pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib


Defaulting to user installation because normal site-packages is not writeable


In [3]:
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import pandas as pd
import io

# Step 1: Set credentials
SERVICE_ACCOUNT_FILE = 'C:\\Users\\naing\\MultiVacSim\\spring25research.json'
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
creds = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=SCOPES
)

# Step 2: Build service
drive_service = build('drive', 'v3', credentials=creds)

# Step 3: Function to load CSV from Drive
def load_csv_from_drive(file_id):
    request = drive_service.files().get_media(fileId=file_id)
    fh = io.BytesIO()
    downloader = MediaIoBaseDownload(fh, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    fh.seek(0)
    return pd.read_csv(fh)

# # Example: COVID hospitalization file
# df = load_csv_from_drive("1euq3iccfFdFX9ipkuX5j-r3z3PtonVBk")
# print(df.head())


In [4]:
from google.oauth2 import service_account 
from googleapiclient.discovery import build

# Step 1: Authenticate with service account
SERVICE_ACCOUNT_FILE = 'C:\\Users\\naing\\MultiVacSim\\spring25research.json'
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
creds = service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)
drive_service = build('drive', 'v3', credentials=creds)

# Step 2: IDs for folders
DATA_FOLDER_ID = "1NyIqr-qkzgxlknIBVN3QPVBmXo9tRJ_P"  # Spring25_Research/data

def list_files_in_folder(folder_id):
    """List all files in a given Google Drive folder."""
    files = []
    page_token = None
    while True:
        response = drive_service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields="nextPageToken, files(id, name, mimeType)",
            pageToken=page_token
        ).execute()
        files.extend(response.get('files', []))
        page_token = response.get('nextPageToken', None)
        if page_token is None:
            break
    return files

def get_subfolder_id(parent_folder_id, folder_name):
    """Get the ID of a subfolder by name."""
    files = list_files_in_folder(parent_folder_id)
    for f in files:
        if f['mimeType'] == 'application/vnd.google-apps.folder' and f['name'].lower() == folder_name.lower():
            return f['id']
    return None

# Step 3: Traverse into 'covid' and 'flu' subfolders
covid_id = get_subfolder_id(DATA_FOLDER_ID, "covid")
flu_id = get_subfolder_id(DATA_FOLDER_ID, "flu")

print("📁 COVID Folder ID:", covid_id)
print("📁 FLU Folder ID:", flu_id)

# Step 4: List CSVs in both folders
def list_csv_files(folder_id):
    files = list_files_in_folder(folder_id)
    return [f for f in files if f['name'].endswith('.csv')]

print("\n✅ CSV Files in COVID Folder:")
for file in list_csv_files(covid_id):
    print(f" - {file['name']} (ID: {file['id']})")

print("\n✅ CSV Files in FLU Folder:")
for file in list_csv_files(flu_id):
    print(f" - {file['name']} (ID: {file['id']})")

📁 COVID Folder ID: 1H7DqlX_6D-iY54EnRNwwzOivCRo_S8jL
📁 FLU Folder ID: 1CMi5zw89YcOa-tDQucLvTxi3Z7la9IrE

✅ CSV Files in COVID Folder:
 - Monthly_Cumulative_Number_and_Percent_of_Persons_Who_Received_1__2024-25_COVID-19_Vaccination_Doses__by_Age_Group__and_Jurisdiction__United_States_20250220.csv (ID: 14kaIvMXiUxm8hsxRFpHpf2vKByGBumbQ)
 - Weekly_Cumulative_Doses__in_Millions__of_Influenza_Vaccines_Distributed_by_Season__United_States_20250220.csv (ID: 1KGw6c3iC1hB3kODpD1fPe1oJjyduwP3P)
 - Weekly_Cumulative_Percentage_of_Children_Ages_6_Months_-17_Years_Who_Are_Up_to_date_with_COVID-19_Vaccines_by_Selected_Demographics_and_by_Season__United_States_20250220.csv (ID: 1_o9UgoGA6eTJFcMhjruAnldGWKEHrp6e)
 - Rates_of_Laboratory-Confirmed_RSV__COVID-19__and_Flu_Hospitalizations_from_the_RESP-NET_Surveillance_Systems_20250220.csv (ID: 1NRDfoVynd8MxYCebHaBmVFmRH369fpIY)
 - Pan-Respiratory_Virus_Emergency_Department_Network_Data_2023-24_National.csv (ID: 1dHfMGgERRhm7npVHiKAh9mlqfVPlWrnM)
 - unite

In [5]:
import pandas as pd

# Re-define after kernel reset
flu_file_ids = {
    "flu_hospitalization": {
        "Laboratory_Confirmed_Influenza_Hospitalizations.csv": "1xx9C4XCr6vF8eLMzKsL2iWtnP8hRxlqu",
        "NHSN_Hospital_Respiratory_Data_2020-21_HHS_Region.csv": "1X8OUeTOKOzMJhffpL4O00GqL_W2qdFGH",
        "NHSN_Hospital_Respiratory_Data_2020-21_National.csv": "1Ggcui1g4Km85DdIVOMJ8528S1Nwl95CC",
        "NHSN_Hospital_Respiratory_Data_2021-22_HHS_Region.csv": "1Qmifhc2dW4GFTT5z24GepthrjuOBw0Eo",
        "NHSN_Hospital_Respiratory_Data_2021-22_National.csv": "1l1E_e7INKjFozKPWyjiEKNw-90xHUF0K",
        "NHSN_Hospital_Respiratory_Data_2022-23_HHS_Region.csv": "1Gi6MMJcFg7mt_tdWHZjYdl3I0Yzz0WVO",
        "NHSN_Hospital_Respiratory_Data_2022-23_National.csv": "15LM7K0d34xYq3hNGcowmhcu9irLzxyVX",
        "NHSN_Hospital_Respiratory_Data_2023-24_HHS_Region.csv": "18wc15duow4ka-bt0a9qKELa8tVNsHcAD",
        "NHSN_Hospital_Respiratory_Data_2023-24_National.csv": "1Xb3RBLB1vmQzWdXmiVYHNoyElPYFfFLA",
        "NHSN_Hospital_Respiratory_Data_2024-25_HHS_Region.csv": "1hLvRartdrMu_C-8OBoyy83sJTgeJmu7r",
        "NHSN_Hospital_Respiratory_Data_2024-25_National.csv": "1QjQMJ27NUxMLLG3thTyC3MUY-rKMiOyq"
    },
    "flu_vaccination": {
        "Characteristics_age.csv": "1aNyi2maXwosGqO5Cjk5cNRnZPZ6gpPhB",
        "Medical_Conditions_age.csv": "18sCBtOSvaqp_HNmTN7xIA6lTMOsLg79p",
        "Weekly_Data_Counts_by_Age.csv": "1TrPfIMu-Oc5I3vB-Rim5qr2YjhsI_SLU",
        "Weekly_Data_Percent_by_Age.csv": "1fsVax5hFtTtF-3nvyZC2bmYybh500D2k",
        "WCD_in_Millions__of_IV_Distributed_by_FS_in_the_US": "1cmxUyiXb-ISzel-8OAh6wiXsZ87OBhhU",
        "IVD Distributed in the United States, By Season": "1PiJ5dXEirrsFELXtNE50XVkPOOJq-cqb",
    },
    "flu_mortality": {
        "PIMS_from_the_NCMSS_National.csv": "1xlwDm2jmrn5NwV45PWMIie_--TBdV8bN",
        "PIMS_from_the_NCMSS_Regional.csv": "150cLlW4BaGyYKEFOh1mdEwIriCOu2VVk",
        "PIMS_from_the_NCMSS_State.csv": "1l8Pyny1_mfXYVcBF55QYRu_oYBPDU6Gh",
        "INFLUENZA_ASSOCIATED_PEDIATRIC_MORTALITY_Weekly.csv": "16ou7A-kbhq_FVg2dLdGw3s8Tug1Uu9_f"
    },
    "flu_trend": {
        "ILINet_hhs_regions.csv": "1ayvMdHuHDgTcZ2VsQymUxQoMv8iVs_Ha",
        "ILINet_national.csv": "1pM86lSyf6hyIGPH-nWgG_SNm_0oP4yM4",
        "ILINet_state.csv": "1Nqh6fE_uZZ_9NSXc1QdqYRlW0Opc12Se",
        "StateDatabySeason48_64.csv": "1U7qy8Gz8hzFdOigU_O49HkNklFYsjB_F",
        "WHO_NREVSS_Clinical_Labs_hhs_regions.csv": "1lXTDp1RtDbNQbecQZgS1x0KRvgk2ll7L",
        "WHO_NREVSS_Clinical_Labs_national.csv": "1DgUo8EGyIHYCgnOvY9xY4xGfyjJTOd0n",
        "WHO_NREVSS_Clinical_Labs_state.csv": "1dVyHeSs3zhNBEAh03LlKL3xVgx003F8e",
        "WHO_NREVSS_Combined_prior_to_2015_16_hhs_regions.csv": "1fK1YT1ZjZKb7tPp2B6RBY2dVt8PGgKNd",
        "WHO_NREVSS_Combined_prior_to_2015_16_national.csv": "1_C85hM-xHJWtOJ7xijzT9NaCA2rHSu6t",
        "WHO_NREVSS_Combined_prior_to_2015_16_state.csv": "1_waiw9cYi5xt_O-I5q7Tmp-In2TOeuGI",
        "WHO_NREVSS_Public_Health_Labs_hhs_regions.csv": "1g24h-DqHQAakLT41gGW_SRDi1GOHa6XU",
        "WHO_NREVSS_Public_Health_Labs_national.csv": "1Mi92JUTzslLz3uWOsqhD1oFiOd-ZzXaC",
        "WHO_NREVSS_Public_Health_Labs_state.csv": "1pdlcKMUSA77h2Y8QjNIcbXTjFRCnUYHN"
    }
}

# 🔍 Keywords per category
relevant_keywords = {
    "flu_hospitalization": ["hospital", "rate", "age", "state", "season"],
    "flu_vaccination": ["vaccine", "vaccination", "dose", "age", "percent"],
    "flu_mortality": ["death", "mortality", "week", "age", "state"],
    "flu_trend": ["percent", "clinic", "outpatient", "ili", "visits", "trend", "week", "positivity", "testing"]
}

# Scoring best file per category
best_files = {}

for category, files in flu_file_ids.items():
    keyword_list = relevant_keywords[category]
    scores = {}
    matches_per_file = {}

    for file_name, file_id in files.items():
        try:
            df = load_csv_from_drive(file_id)
            columns = df.columns.str.lower()
            matched_keywords = [kw for kw in keyword_list if any(kw in col for col in columns)]
            scores[file_name] = len(matched_keywords)
            matches_per_file[file_name] = matched_keywords
        except Exception as e:
            scores[file_name] = -1
            matches_per_file[file_name] = [f"❌ Error: {str(e)}"]

    best_file = max(scores, key=scores.get)
    best_files[category] = {
        "best_file": best_file,
        "score": scores[best_file],
        "matched_keywords": matches_per_file[best_file],
        "other_files": {f: {"score": s, "matched_keywords": matches_per_file[f]} for f, s in scores.items() if f != best_file}
    }

# 📊 Output report
print("\n📊 BEST FILES PER CATEGORY (Based on Keyword Matching)\n")
for category, data in best_files.items():
    print(f"🔹 Category: {category}")
    print(f"✅ Best File: {data['best_file']}")
    print(f"   ➤ Score (matched keywords): {data['score']}")
    print(f"   ➤ Keywords used: {', '.join(relevant_keywords[category])}")
    print(f"   ➤ Matched in best file: {', '.join(data['matched_keywords']) if data['matched_keywords'] else '(none)'}")

    print("   📁 Other Files:")
    for file, info in data["other_files"].items():
        print(f"     - {file} → Score: {info['score']}, Matched: {', '.join(info['matched_keywords']) if info['matched_keywords'] else '(none)'}")

    print("-" * 80)


C:\Users\naing\AppData\Local\Temp\ipykernel_19692\3950639955.py:26: DtypeWarning: Columns (3,4,9) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(fh)



📊 BEST FILES PER CATEGORY (Based on Keyword Matching)

🔹 Category: flu_hospitalization
✅ Best File: NHSN_Hospital_Respiratory_Data_2021-22_HHS_Region.csv
   ➤ Score (matched keywords): 2
   ➤ Keywords used: hospital, rate, age, state, season
   ➤ Matched in best file: hospital, rate
   📁 Other Files:
     - Laboratory_Confirmed_Influenza_Hospitalizations.csv → Score: 0, Matched: (none)
     - NHSN_Hospital_Respiratory_Data_2020-21_HHS_Region.csv → Score: 1, Matched: hospital
     - NHSN_Hospital_Respiratory_Data_2020-21_National.csv → Score: 1, Matched: age
     - NHSN_Hospital_Respiratory_Data_2021-22_National.csv → Score: 1, Matched: hospital
     - NHSN_Hospital_Respiratory_Data_2022-23_HHS_Region.csv → Score: 1, Matched: hospital
     - NHSN_Hospital_Respiratory_Data_2022-23_National.csv → Score: 2, Matched: hospital, rate
     - NHSN_Hospital_Respiratory_Data_2023-24_HHS_Region.csv → Score: 2, Matched: hospital, rate
     - NHSN_Hospital_Respiratory_Data_2023-24_National.csv → Sc

In [6]:
import os
# ---- File IDs ----
nhsn_files = {
    "HHS_Region": [
        "1X8OUeTOKOzMJhffpL4O00GqL_W2qdFGH",
        "1Qmifhc2dW4GFTT5z24GepthrjuOBw0Eo",
        "1Gi6MMJcFg7mt_tdWHZjYdl3I0Yzz0WVO",
        "18wc15duow4ka-bt0a9qKELa8tVNsHcAD",
        "1hLvRartdrMu_C-8OBoyy83sJTgeJmu7r"
    ],
    "National": [
        "1Ggcui1g4Km85DdIVOMJ8528S1Nwl95CC",
        "1l1E_e7INKjFozKPWyjiEKNw-90xHUF0K",
        "15LM7K0d34xYq3hNGcowmhcu9irLzxyVX",
        "1Xb3RBLB1vmQzWdXmiVYHNoyElPYFfFLA",
        "1QjQMJ27NUxMLLG3thTyC3MUY-rKMiOyq"
    ]
}

# ---- Output Folder ----
output_path = "C:\\Users\\naing\\MultiVacSim\\data\\flu\\processed"

# ---- Process and Save ----
REQUIRED_COL_COUNT = 6

for category, file_ids in nhsn_files.items():
    dfs = []
    for fid in file_ids:
        try:
            print(f"📥 Loading file from ID: {fid}")
            df = load_csv_from_drive(fid)
            if df.shape[1] != REQUIRED_COL_COUNT:
                print(f"⚠️ Warning: File with ID {fid} has {df.shape[1]} columns. Trimming or padding to 6.")
                if df.shape[1] > REQUIRED_COL_COUNT:
                    df = df.iloc[:, :REQUIRED_COL_COUNT]
                else:
                    for i in range(REQUIRED_COL_COUNT - df.shape[1]):
                        df[f"col_pad_{i+1}"] = None
            dfs.append(df)
        except Exception as e:
            print(f"❌ Error loading file {fid}: {e}")
    
    if dfs:
        # Align column names to those of the first valid dataframe
        base_columns = dfs[0].columns
        for i in range(1, len(dfs)):
            dfs[i].columns = base_columns
        
        combined_df = pd.concat(dfs, ignore_index=True)
        output_file = os.path.join(output_path, f"NHSN_{category}.csv")
        combined_df.to_csv(output_file, index=False)
        print(f"✅ Saved: {output_file}")

📥 Loading file from ID: 1X8OUeTOKOzMJhffpL4O00GqL_W2qdFGH
📥 Loading file from ID: 1Qmifhc2dW4GFTT5z24GepthrjuOBw0Eo
📥 Loading file from ID: 1Gi6MMJcFg7mt_tdWHZjYdl3I0Yzz0WVO
📥 Loading file from ID: 18wc15duow4ka-bt0a9qKELa8tVNsHcAD
📥 Loading file from ID: 1hLvRartdrMu_C-8OBoyy83sJTgeJmu7r
✅ Saved: C:\Users\naing\MultiVacSim\data\flu\processed\NHSN_HHS_Region.csv
📥 Loading file from ID: 1Ggcui1g4Km85DdIVOMJ8528S1Nwl95CC
📥 Loading file from ID: 1l1E_e7INKjFozKPWyjiEKNw-90xHUF0K
📥 Loading file from ID: 15LM7K0d34xYq3hNGcowmhcu9irLzxyVX
📥 Loading file from ID: 1Xb3RBLB1vmQzWdXmiVYHNoyElPYFfFLA
📥 Loading file from ID: 1QjQMJ27NUxMLLG3thTyC3MUY-rKMiOyq
✅ Saved: C:\Users\naing\MultiVacSim\data\flu\processed\NHSN_National.csv
